Load dataset from the class LCAmazon in train mode and its data augmented version

In [1]:
from models.lcamazon import LCAmazon


dataset=LCAmazon(root="DATA", modality="s2", split="train")
print(f"dataset of length {len(dataset)}")
aug=LCAmazon(root="DATA", modality="s2", split="train", aug_geometric=True)
print(f"with data augmentation length {len(aug)}")

dataset of length 3840
with data augmentation length 3840


Checking shape of image and label mask (ground truth)

In [2]:
import numpy as np
img, label = dataset[0]
print(f"Image of shape: {np.shape(img)}, Label mask (ground truth) of shape: {np.shape(label)}")

Image of shape: (47, 47, 12), Label mask (ground truth) of shape: (47, 47)


Let's now plot the satellite image together with groundtruth. We also plot the data augmented version below to check if both the groundtruth and the image are transformed correctly.

In [3]:
from ipywidgets import interact
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np


@interact(idx=range(len(dataset)))
def plot_sample(idx=0):
    # Original sample
    img, label = dataset[idx]

    r,g,b   = img[:, :, 3], img[:, :, 2], img[:, :, 1]
    rgb = np.stack([r,g,b], axis=-1).astype(np.float32)
    rgb /= np.percentile(rgb, 99)
    rgb = np.clip(rgb, 0, 1)

    # Augmented sample
    aug_img, aug_label = aug[idx]

    r_aug, g_aug, b_aug   = aug_img[:, :, 3], aug_img[:, :, 2], aug_img[:, :, 1]
    rgb_aug = np.stack([r_aug, g_aug, b_aug], axis=-1).astype(np.float32)
    rgb_aug /= np.percentile(rgb_aug, 99)
    rgb_aug = np.clip(rgb_aug, 0, 1)

    # Class mapping
    class_mapping = {
        new_id: class_name       # to deal with unused classes
        for class_name, old_id in LCAmazon.LABEL_CLASSES.items()
        if old_id in LCAmazon.LABEL_REMAP
        for new_id in [LCAmazon.LABEL_REMAP[old_id]]
    }

    unique_labels = np.unique(label)
    cmap = plt.cm.get_cmap("tab20", len(class_mapping))

    # Plotting
    fig, axes = plt.subplots(2, 2, figsize=(6, 6))

    # Original RGB
    axes[0, 0].imshow(rgb)
    axes[0, 0].set_title("Original RGB")
    axes[0, 0].axis("off")

    # Original GT
    axes[0, 1].imshow(label, cmap=cmap, vmin=0, vmax=len(class_mapping)-1)
    axes[0, 1].set_title("Original Ground Truth")
    axes[0, 1].axis("off")

    # Augmented RGB
    axes[1, 0].imshow(rgb_aug)
    axes[1, 0].set_title("Augmented RGB")
    axes[1, 0].axis("off")

    # Augmented GT
    axes[1, 1].imshow(aug_label, cmap=cmap, vmin=0, vmax=len(class_mapping)-1)
    axes[1, 1].set_title("Augmented Ground Truth")
    axes[1, 1].axis("off")

    # Legend
    legend_patches = [
        mpatches.Patch(color=cmap(class_id), label=class_mapping[class_id])
        for class_id in unique_labels
    ]

    axes[1, 1].legend(
        handles=legend_patches,
        bbox_to_anchor=(1.05, 1),
        loc="upper left",
        borderaxespad=0.
    )

    plt.tight_layout()
    plt.show()


interactive(children=(Dropdown(description='idx', options=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 1…

Qualitative assessment of predictions from the Sentinel-2 model --> train dataset

In [7]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import rasterio
from ipywidgets import interact

REMAPPED_ID_TO_NAME = {
    LCAmazon.LABEL_REMAP[v]: k
    for k, v in LCAmazon.LABEL_CLASSES.items()
}

N_CLASSES = max(REMAPPED_ID_TO_NAME.keys()) 
CMAP = plt.cm.get_cmap("tab20", N_CLASSES + 1)
PRED_ROOT = "modeloutputs/s2_prediction"

@interact(idx=range(len(dataset)))
def plot_all(idx=0):
    img, gt_label = dataset[idx]

    # RGB
    r,g,b   = img[:, :, 3], img[:, :, 2], img[:, :, 1]
    rgb = np.stack([r,g,b], axis=-1).astype(np.float32)
    rgb /= np.percentile(rgb, 99)
    rgb = np.clip(rgb, 0, 1)

    # Prediction
    _, gt_path = dataset.samples[idx]
    fname = os.path.basename(gt_path)
    pred_path = os.path.join(PRED_ROOT, fname)

    if not os.path.exists(pred_path):
        raise FileNotFoundError(f"Prediction not found: {pred_path}")

    with rasterio.open(pred_path) as src:
        pred_label = src.read(1).astype(np.int32)

    # Plotting
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    # RGB
    axes[0].imshow(rgb)
    axes[0].set_title("RGB")
    axes[0].axis("off")
    # Ground Truth
    axes[1].imshow(gt_label, cmap=CMAP, vmin=0, vmax=N_CLASSES)
    axes[1].set_title("Ground Truth")
    axes[1].axis("off")
    # Prediction
    axes[2].imshow(pred_label, cmap=CMAP, vmin=0, vmax=N_CLASSES)
    axes[2].set_title("Prediction")
    axes[2].axis("off")
    # Legend
    unique_labels = np.unique(
        np.concatenate([np.unique(gt_label), np.unique(pred_label)])
    )
    
    legend_patches = []
    for class_id in unique_labels:
        if class_id == 0:
            name = "Background / Ignored"
        else:
            name = REMAPPED_ID_TO_NAME.get(class_id, f"Unknown ({class_id})")

        legend_patches.append(
            mpatches.Patch(color=CMAP(class_id), label=name)
        )

    fig.legend(
        handles=legend_patches,
        bbox_to_anchor=(1.05, 0.5),
        loc="center left"
    )

    plt.tight_layout()
    plt.show()


/tmp/ipykernel_3117359/3181417913.py:14: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  CMAP = plt.cm.get_cmap("tab20", N_CLASSES + 1)


interactive(children=(Dropdown(description='idx', options=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 1…

Qualitative assessment of predictions from the Sentinel-2 model --> test dataset

In [11]:
import os
import rasterio
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from ipywidgets import interact

test_dataset = LCAmazon(root="DATA", modality="s2", split="test")

REMAPPED_ID_TO_NAME = {
    LCAmazon.LABEL_REMAP[v]: k
    for k, v in LCAmazon.LABEL_CLASSES.items()
}

N_CLASSES = max(REMAPPED_ID_TO_NAME.keys()) 
CMAP = plt.cm.get_cmap("tab20", N_CLASSES + 1)
PRED_ROOT = "modeloutputs/s2_prediction"

@interact(idx=range(len(test_dataset)))
def plot_all(idx=0):
    img, gt_label = test_dataset[idx]

    # RGB
    r,g,b   = img[:, :, 3], img[:, :, 2], img[:, :, 1]
    rgb = np.stack([r,g,b], axis=-1).astype(np.float32)
    rgb /= np.percentile(rgb, 99)
    rgb = np.clip(rgb, 0, 1)

    # Prediction
    _, gt_path = test_dataset.samples[idx]
    fname = os.path.basename(gt_path)
    pred_path = os.path.join(PRED_ROOT, fname)

    if not os.path.exists(pred_path):
        raise FileNotFoundError(f"Prediction not found: {pred_path}")

    with rasterio.open(pred_path) as src:
        pred_label = src.read(1).astype(np.int32)

    # Plotting
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    # RGB
    axes[0].imshow(rgb)
    axes[0].set_title("RGB")
    axes[0].axis("off")
    # Ground Truth
    axes[1].imshow(gt_label, cmap=CMAP, vmin=0, vmax=N_CLASSES)
    axes[1].set_title("Ground Truth")
    axes[1].axis("off")
    # Prediction
    axes[2].imshow(pred_label, cmap=CMAP, vmin=0, vmax=N_CLASSES)
    axes[2].set_title("Prediction")
    axes[2].axis("off")
    # Legend
    unique_labels = np.unique(
        np.concatenate([np.unique(gt_label), np.unique(pred_label)])
    )
    
    legend_patches = []
    for class_id in unique_labels:
        if class_id == 0:
            name = "Background / Ignored"
        else:
            name = REMAPPED_ID_TO_NAME.get(class_id, f"Unknown ({class_id})")

        legend_patches.append(
            mpatches.Patch(color=CMAP(class_id), label=name)
        )

    fig.legend(
        handles=legend_patches,
        bbox_to_anchor=(1.05, 0.5),
        loc="center left"
    )

    plt.tight_layout()
    plt.show()


/tmp/ipykernel_3117359/2791890589.py:16: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  CMAP = plt.cm.get_cmap("tab20", N_CLASSES + 1)


interactive(children=(Dropdown(description='idx', options=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 1…

NOW FOR THE MODEL BASED ON AE EMBEDDINGS (Maxime : still need to fix the function saving images to modeloutputs/AE_prediction)

In [7]:
import os
import rasterio
REMAPPED_ID_TO_NAME = {
    LCAmazon.LABEL_REMAP[v]: k
    for k, v in LCAmazon.LABEL_CLASSES.items()
}

N_CLASSES = max(REMAPPED_ID_TO_NAME.keys())  # 12
CMAP = plt.cm.get_cmap("tab20", N_CLASSES + 1)
PRED_ROOT = "modeloutputs/AE_prediction"

@interact(idx=range(len(dataset)))
def plot_gt_vs_pred(idx=0):
    img, gt_label = dataset[idx]

    # Get filename from dataset
    _, gt_path = dataset.samples[idx]
    fname = os.path.basename(gt_path)

    pred_path = os.path.join(PRED_ROOT, fname)

    if not os.path.exists(pred_path):
        raise FileNotFoundError(f"Prediction not found: {pred_path}")

    # Load prediction
    with rasterio.open(pred_path) as src:
        pred_label = src.read(1).astype(np.int32)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    axes[0].imshow(gt_label, cmap=CMAP, vmin=0, vmax=N_CLASSES)
    axes[0].set_title("Ground Truth")
    axes[0].axis("off")

    axes[1].imshow(pred_label, cmap=CMAP, vmin=0, vmax=N_CLASSES)
    axes[1].set_title("Prediction")
    axes[1].axis("off")

    # Legend (union of GT and prediction labels)
    unique_labels = np.unique(
        np.concatenate([np.unique(gt_label), np.unique(pred_label)])
    )

    legend_patches = []
    for class_id in unique_labels:
        if class_id == 0:
            name = "Background / Ignored"
        else:
            name = REMAPPED_ID_TO_NAME.get(class_id, f"Unknown ({class_id})")

        legend_patches.append(
            mpatches.Patch(color=CMAP(class_id), label=name)
        )

    fig.legend(
        handles=legend_patches,
        bbox_to_anchor=(1.05, 0.5),
        loc="center left"
    )

    plt.tight_layout()
    plt.show()


/tmp/ipykernel_1020463/2445333875.py:9: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  CMAP = plt.cm.get_cmap("tab20", N_CLASSES + 1)


interactive(children=(Dropdown(description='idx', options=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 1…